<a href="https://colab.research.google.com/github/abhi460729/Generative-AI-and-LLM/blob/main/BERT_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentence classification using Transfer Learning with Huggingface BERT and Weights and Biases

This notebook fine-tunes a BERT model on the CoLA dataset for single sentence classification, determining if sentences are grammatically correct or incorrect. We use Hugging Face Transformers for the model and tokenizer, and Weights & Biases (W&B) for logging metrics. The training is limited to 2 epochs with 10 training batches and 5 validation batches per epoch to stop after a few iterations.

## Install Dependencies

In [37]:
!pip install transformers wandb pandas numpy torch wget

## Download the CoLA Dataset

We’ll use The Corpus of Linguistic Acceptability (CoLA) dataset for single sentence classification. It’s a set of sentences labeled as grammatically correct or incorrect.

In [38]:
import wget
import os

print('Downloading dataset...')

# The URL for the dataset zip file.
url = 'https://nyu-mll.github.io/CoLA/cola_public_1.1.zip'

# Download the file (if we haven't already)
if not os.path.exists('./cola_public_1.1.zip'):
    wget.download(url, './cola_public_1.1.zip')

if not os.path.exists('./cola_public/'):
    !unzip cola_public_1.1.zip

## Load and Inspect the Dataset

In [39]:
import pandas as pd

# Load the dataset into a pandas dataframe.
df = pd.read_csv("./cola_public/raw/in_domain_train.tsv", delimiter='\t', header=None, names=['sentence_source', 'label', 'label_notes', 'sentence'])

# Report the number of sentences.
print('Number of training sentences: {:,}\n'.format(df.shape[0]))

# Display 10 random rows from the data.
df.sample(10)

Number of training sentences: 8,551



,sentence_source,label,label_notes,sentence
2389,l-93,1,NaN,Angela characterized Shelly as a lifesaver.
5048,ks08,1,NaN,They're not finding it a stress being in the s...
3133,l-93,0,*,Paul exhaled on Mary.
5955,c_13,0,*,I ordered if John drink his beer.
625,bc01,1,NaN,Press the stamp against the pad completely.
3542,ks08,0,*,They can very.
6915,m_02,1,NaN,This arch is supporting the weight of the tower.
2908,l-93,1,NaN,That new handle detaches easily.
5857,c_13,1,NaN,The Brazilians pumped the oil across the river.
4191,ks08,1,NaN,It is a wooden desk.


## Tokenization

We use the BERT tokenizer to preprocess the sentences, adding special tokens (`[CLS]` and `[SEP]`), padding/truncating to a fixed length, and creating attention masks.

In [40]:
from transformers import BertTokenizer

# Load the BERT tokenizer.
print('Loading BERT tokenizer...')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

# Get the lists of sentences and their labels.
sentences = df.sentence.values
labels = df.label.values

# Find the maximum sentence length
max_len = 0
for sent in sentences:
    input_ids = tokenizer.encode(sent, add_special_tokens=True)
    max_len = max(max_len, len(input_ids))
print('Max sentence length: ', max_len)

Loading BERT tokenizer...
Max sentence length:  47


## Data Preparation

Tokenize the dataset, create attention masks, and set up data loaders for training and validation.

In [41]:
def ret_dataloader(max_len=64, batch_size=32):
    from torch.utils.data import DataLoader, TensorDataset, RandomSampler, SequentialSampler

    # Drop NaNs from source DataFrame
    global df
    df = df.dropna(subset=['sentence', 'label'])

    sentences = df.sentence.values
    labels = df.label.values

    # Encode
    encodings = tokenizer.batch_encode_plus(
        list(sentences),
        add_special_tokens=True,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

    input_ids = encodings['input_ids']
    attention_masks = encodings['attention_mask']
    labels_tensor = torch.tensor(labels)

    # Sanity check
    assert input_ids.shape[0] == labels_tensor.shape[0] == attention_masks.shape[0], \
        f"Tensor size mismatch: input_ids={input_ids.shape}, attention_masks={attention_masks.shape}, labels={labels_tensor.shape}"

    dataset = TensorDataset(input_ids, attention_masks, labels_tensor)

    # Split
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

    train_dataloader = DataLoader(train_dataset, sampler=RandomSampler(train_dataset), batch_size=batch_size)
    validation_dataloader = DataLoader(val_dataset, sampler=SequentialSampler(val_dataset), batch_size=batch_size)

    return train_dataloader, validation_dataloader


## Model, Optimizer, and Scheduler

Define functions to load the BERT model, optimizer, and learning rate scheduler.

In [42]:
from transformers import BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

def ret_model():
    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
    return model

def ret_optim(model):
    return AdamW(model.parameters(), lr=5e-5, eps=1e-8)

def ret_scheduler(train_dataloader, optimizer, epochs):
    total_steps = len(train_dataloader) * epochs
    return get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)


## Utility Functions

Define helper functions for formatting time and calculating accuracy.

In [43]:
import numpy as np
import time

def format_time(elapsed):
    return str(int(elapsed // 60)) + 'm' + str(int(elapsed % 60)) + 's'

def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

## Training Loop

Define the training function, which runs for 2 epochs with limited batches to stop quickly.

In [44]:
import wandb
import random

def train():
    # Initialize W&B
    wandb.init(project="cola-bert", config={
        "epochs": 4,                 # Increase epochs
        "learning_rate": 2e-5,
        "batch_size": 32,
        "max_batches_per_epoch": 200,  # <-- Increase from 10
        "max_val_batches": 50
    })

    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Load model and move to device
    model = ret_model().to(device)

    # Load data loaders and optimizer
    train_dataloader, validation_dataloader = ret_dataloader()
    optimizer = ret_optim(model)
    scheduler = ret_scheduler(train_dataloader, optimizer, epochs=wandb.config.epochs)

    # Set random seed
    seed_val = 42
    random.seed(seed_val)
    np.random.seed(seed_val)
    torch.manual_seed(seed_val)
    if device.type == 'cuda':
        torch.cuda.manual_seed_all(seed_val)

    training_stats = []
    total_t0 = time.time()
    epochs = wandb.config.epochs
    max_batches_per_epoch = wandb.config.max_batches_per_epoch
    max_val_batches = wandb.config.max_val_batches

    for epoch_i in range(epochs):
        print(f"\n======== Epoch {epoch_i + 1} / {epochs} ========")
        print("Training...")

        t0 = time.time()
        total_train_loss = 0
        model.train()

        for step, batch in enumerate(train_dataloader):
            if step >= max_batches_per_epoch:
                break
            if step % 40 == 0 and step != 0:
                elapsed = format_time(time.time() - t0)
                print(f"  Batch {step:>5,} of {max_batches_per_epoch:>5,}. Elapsed: {elapsed}.")

            b_input_ids = batch[0].to(device)
            b_input_mask = batch[1].to(device)
            b_labels = batch[2].to(device)

            model.zero_grad()
            outputs = model(
                b_input_ids,
                token_type_ids=None,
                attention_mask=b_input_mask,
                labels=b_labels
            )

            loss = outputs.loss
            logits = outputs.logits

            wandb.log({'train_batch_loss': loss.item()})
            total_train_loss += loss.item()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        avg_train_loss = total_train_loss / max_batches_per_epoch
        training_time = format_time(time.time() - t0)
        wandb.log({'avg_train_loss': avg_train_loss})

        print(f"  Average training loss: {avg_train_loss:.2f}")
        print(f"  Training epoch took: {training_time}")

        print("\nRunning Validation...")
        t0 = time.time()
        model.eval()
        total_eval_accuracy = 0
        total_eval_loss = 0

        for i, batch in enumerate(validation_dataloader):
            if i >= max_val_batches:
                break
            b_input_ids = batch[0].to(device)
            b_input_mask = batch[1].to(device)
            b_labels = batch[2].to(device)

            with torch.no_grad():
                outputs = model(
                    b_input_ids,
                    token_type_ids=None,
                    attention_mask=b_input_mask,
                    labels=b_labels
                )

                loss = outputs.loss
                logits = outputs.logits

            total_eval_loss += loss.item()
            logits = logits.detach().cpu().numpy()
            label_ids = b_labels.to('cpu').numpy()
            total_eval_accuracy += flat_accuracy(logits, label_ids)

        avg_val_accuracy = total_eval_accuracy / max_val_batches
        avg_val_loss = total_eval_loss / max_val_batches
        validation_time = format_time(time.time() - t0)

        wandb.log({'val_accuracy': avg_val_accuracy, 'avg_val_loss': avg_val_loss})
        print(f"  Accuracy: {avg_val_accuracy:.2f}")
        print(f"  Validation Loss: {avg_val_loss:.2f}")
        print(f"  Validation took: {validation_time}")

        training_stats.append({
            'epoch': epoch_i + 1,
            'Training Loss': avg_train_loss,
            'Valid. Loss': avg_val_loss,
            'Valid. Accur.': avg_val_accuracy,
            'Training Time': training_time,
            'Validation Time': validation_time
        })

    print("\nTraining complete!")
    print(f"Total training took {format_time(time.time() - total_t0)} (h:mm:ss)")
    wandb.finish()

    return model, training_stats


## Run Training

Execute the training loop.

In [45]:
model, training_stats = train()

Using device: cpu


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



======== Epoch 1 / 4 ========
Training...
  Batch    40 of   200. Elapsed: 15m21s.
  Batch    80 of   200. Elapsed: 29m49s.
  Batch   120 of   200. Elapsed: 44m18s.
  Batch   160 of   200. Elapsed: 58m42s.
  Average training loss: 0.51
  Training epoch took: 73m5s

Running Validation...
  Accuracy: 0.81
  Validation Loss: 0.44
  Validation took: 5m42s

======== Epoch 2 / 4 ========
Training...
  Batch    40 of   200. Elapsed: 14m24s.
  Batch    80 of   200. Elapsed: 28m44s.
  Batch   120 of   200. Elapsed: 43m1s.
  Batch   160 of   200. Elapsed: 57m21s.
  Average training loss: 0.31
  Training epoch took: 71m42s

Running Validation...
  Accuracy: 0.82
  Validation Loss: 0.42
  Validation took: 5m33s

======== Epoch 3 / 4 ========
Training...
  Batch    40 of   200. Elapsed: 14m18s.
  Batch    80 of   200. Elapsed: 28m39s.
  Batch   120 of   200. Elapsed: 42m58s.
  Batch   160 of   200. Elapsed: 57m21s.
  Average training loss: 0.19
  Training epoch took: 71m40s

Running Validation...


avg_train_loss,█▅▂▁
avg_val_loss,▁▁▄█
train_batch_loss,██▆▇▇▆▇▆▆▇▅▆▆▅▆▃▄▄▄▃▄▂▄▅▄▅▃▄▃▂▁▂▃▂▄▁▃▃▄▃
val_accuracy,▁▅▄█
avg_train_loss,0.11135
avg_val_loss,0.67276
train_batch_loss,0.01422
val_accuracy,0.8375


In [46]:
def predict_grammar(sentences, model, tokenizer, device=None, max_len=64):
    """
    Predict grammatical correctness of sentences using a fine-tuned BERT model on CoLA.

    Args:
        sentences (list[str]): List of sentences to evaluate.
        model (BertForSequenceClassification): Trained model.
        tokenizer (BertTokenizer): Tokenizer used for preprocessing.
        device (torch.device): CUDA or CPU.
        max_len (int): Maximum token length (default: 64).

    Returns:
        List of dicts with sentence, prediction (0/1), and confidence.
    """
    if isinstance(sentences, str):
        sentences = [sentences]

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.eval()
    model.to(device)

    # Tokenize and encode
    encodings = tokenizer.batch_encode_plus(
        sentences,
        add_special_tokens=True,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

    input_ids = encodings['input_ids'].to(device)
    attention_mask = encodings['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1).cpu().numpy()
        confs = probs[:, 1].cpu().numpy()  # Confidence of "grammatical" class

    # Format result
    results = []
    for sent, pred, conf in zip(sentences, preds, confs):
        results.append({
            "sentence": sent,
            "is_grammatical": bool(pred),
            "confidence": float(conf)
        })

    return results

In [47]:
# Assume model, tokenizer already loaded and trained
sentences = [
    "He go to school yesterday.",                     # Wrong verb tense ("go" → "went")
    "They going to the park now.",                    # Missing auxiliary verb ("are going")
    "The book is on table.",                          # Missing article ("the table")
    "She likes read novels.",                         # Wrong verb form ("read" → "to read" or "reading")
    "Is raining today.",                              # Missing subject ("It is raining")
    "A dog bark at night.",                           # Verb agreement ("barks")
    "The quickly fox jumped over lazy dog.",          # Incorrect adjective/adverb placement
    "She can sings very well.",                       # Modal + base form ("sing" not "sings")
    "Him go to the party.",                           # Wrong pronoun case ("He" not "Him")
    "What means this?",                               # Word order ("What does this mean?")
]

results = predict_grammar(sentences, model, tokenizer)

for r in results:
    print(f"Sentence: {r['sentence']}")
    print(f"✅ Grammatical: {r['is_grammatical']}, Confidence: {r['confidence']:.2f}\n")

Sentence: He go to school yesterday.
✅ Grammatical: False, Confidence: 0.03

Sentence: They going to the park now.
✅ Grammatical: False, Confidence: 0.03

Sentence: The book is on table.
✅ Grammatical: True, Confidence: 0.98

Sentence: She likes read novels.
✅ Grammatical: False, Confidence: 0.16

Sentence: Is raining today.
✅ Grammatical: True, Confidence: 0.99

Sentence: A dog bark at night.
✅ Grammatical: True, Confidence: 0.95

Sentence: The quickly fox jumped over lazy dog.
✅ Grammatical: False, Confidence: 0.02

Sentence: She can sings very well.
✅ Grammatical: False, Confidence: 0.03

Sentence: Him go to the party.
✅ Grammatical: False, Confidence: 0.03

Sentence: What means this?
✅ Grammatical: True, Confidence: 1.00

